# 01 — Model Analysis

This notebook loads a trained `MLEngine` model from disk and inspects:
1. The autoencoder's weight matrices (heatmaps)
2. The training reconstruction-error distribution vs. the anomaly threshold
3. The scaler's learned mean/std per feature

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python-backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0e14'
matplotlib.rcParams['axes.facecolor'] = '#12161f'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#8b949e'
matplotlib.rcParams['xtick.color'] = '#8b949e'
matplotlib.rcParams['ytick.color'] = '#8b949e'

from ml_engine import MLEngine, FEATURE_NAMES, FEATURE_DIM

In [ ]:
# Load the persisted model
MODEL_PATH = os.path.join('..', 'python-backend', 'model.pkl')

engine = MLEngine(input_dim=FEATURE_DIM)

if os.path.exists(MODEL_PATH):
    engine.load(MODEL_PATH)
    print(f'Loaded model. Status: {engine.get_status()}')
    print(f'Training metadata: {engine.get_training_metadata()}')
else:
    print('No model.pkl found. Run the backend and let it train first.')
    print(f'Looked at: {os.path.abspath(MODEL_PATH)}')

## Weight Matrix Heatmaps

Visualize what the autoencoder learned by plotting W1 (encoder) and W4 (decoder).

In [ ]:
if engine._model is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    im1 = axes[0].imshow(engine._model.W1, aspect='auto', cmap='coolwarm')
    axes[0].set_title('W1 (Input → Hidden)', fontsize=14, pad=12)
    axes[0].set_ylabel('Input Features')
    axes[0].set_xlabel('Hidden Units')
    axes[0].set_yticks(range(len(FEATURE_NAMES)))
    axes[0].set_yticklabels(FEATURE_NAMES, fontsize=8)
    plt.colorbar(im1, ax=axes[0], shrink=0.8)
    
    im2 = axes[1].imshow(engine._model.W4.T, aspect='auto', cmap='coolwarm')
    axes[1].set_title('W4 (Hidden → Output / Decoder)', fontsize=14, pad=12)
    axes[1].set_ylabel('Output Features')
    axes[1].set_xlabel('Hidden Units')
    axes[1].set_yticks(range(len(FEATURE_NAMES)))
    axes[1].set_yticklabels(FEATURE_NAMES, fontsize=8)
    plt.colorbar(im2, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print('Model not loaded — cannot plot weights.')

## Scaler Statistics

The scaler's learned mean and standard deviation per feature tell us which features had the most variance during the Grace Period.

In [ ]:
if engine._scaler._fitted:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    y_pos = np.arange(len(FEATURE_NAMES))
    
    axes[0].barh(y_pos, engine._scaler.mean_, color='#58a6ff', height=0.6)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(FEATURE_NAMES, fontsize=8)
    axes[0].set_title('Feature Means (Scaler)', fontsize=14, pad=12)
    axes[0].invert_yaxis()
    
    axes[1].barh(y_pos, engine._scaler.std_, color='#bc8cff', height=0.6)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(FEATURE_NAMES, fontsize=8)
    axes[1].set_title('Feature Std Deviations (Scaler)', fontsize=14, pad=12)
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print('Scaler not fitted yet.')

## Reconstruction Error Distribution

Simulate normal and anomalous traffic to visualize the error distribution and threshold.

In [ ]:
if engine._model is not None and engine._scaler._fitted:
    rng = np.random.default_rng(42)
    # Generate synthetic normal samples near the scaler's mean
    normal_samples = engine._scaler.mean_ + rng.normal(0, engine._scaler.std_ * 0.5, size=(200, FEATURE_DIM))
    normal_samples = np.clip(normal_samples, 0, None)
    
    # Generate synthetic anomalous samples far from the mean
    anomaly_samples = engine._scaler.mean_ + rng.normal(0, engine._scaler.std_ * 3.0, size=(50, FEATURE_DIM))
    anomaly_samples = np.clip(anomaly_samples, 0, None)
    
    normal_errors = [engine.score_packet(s) for s in normal_samples]
    anomaly_errors = [engine.score_packet(s) for s in anomaly_samples]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(normal_errors, bins=40, alpha=0.7, color='#3fb950', label='Normal Traffic')
    ax.hist(anomaly_errors, bins=20, alpha=0.7, color='#f85149', label='Anomalous Traffic')
    if engine._threshold:
        ax.axvline(engine._threshold, color='#f0883e', linestyle='--', linewidth=2, label=f'Threshold ({engine._threshold:.4f})')
    ax.set_xlabel('Reconstruction Error')
    ax.set_ylabel('Frequency')
    ax.set_title('Reconstruction Error Distribution: Normal vs. Anomalous', fontsize=14, pad=12)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Model not loaded — cannot compute errors.')